# Traffic Demand Prediction — Version Alpha

Self-contained notebook for the fold-safe ensemble pipeline.
It reads `dataset/train.csv` and `dataset/test.csv` and writes `submission_alpha.csv`.


In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor


RANDOM_STATE = 42
N_SPLITS = 5
BASE_DIR = Path.cwd()
TRAIN_PATH = BASE_DIR / 'dataset' / 'train.csv'
TEST_PATH = BASE_DIR / 'dataset' / 'test.csv'
OUTPUT_PATH = BASE_DIR / 'submission_alpha.csv'


@dataclass
class Config:
    train_path: Path
    test_path: Path
    output_path: Path


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ts_parts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour'] = ts_parts[0].astype(int)
    df['minute'] = ts_parts[1].astype(int)
    df['time_minutes'] = df['hour'] * 60 + df['minute']
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['time_sin'] = np.sin(2 * np.pi * df['time_minutes'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['time_minutes'] / 1440)
    df['rush_hour'] = (((df['hour'] >= 7) & (df['hour'] <= 10)) | ((df['hour'] >= 17) & (df['hour'] <= 20))).astype(int)
    df['night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['midday'] = ((df['hour'] >= 11) & (df['hour'] <= 16)).astype(int)
    df['time_period'] = pd.cut(
        df['hour'],
        bins=[-1, 5, 10, 15, 20, 24],
        labels=['night', 'morning', 'midday', 'evening', 'late_night'],
    )
    df['geo_prefix'] = df['geohash'].astype(str).str[:4]
    df['geo_prefix3'] = df['geohash'].astype(str).str[:3]
    df['geo_prefix5'] = df['geohash'].astype(str).str[:5]
    df['temp_missing'] = df['Temperature'].isnull().astype(int)
    df['road_missing'] = df['RoadType'].isnull().astype(int)
    df['temp_bin'] = pd.cut(
        df['Temperature'],
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=['cold', 'mild', 'warm', 'hot'],
    )
    df['temp_squared'] = df['Temperature'] ** 2
    df['hour_squared'] = df['hour'] ** 2
    df['day_x_hour'] = df['day'] * df['hour']
    return df


def encode_categoricals(train: pd.DataFrame, test: pd.DataFrame, categorical_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat([train, test], axis=0, ignore_index=True)
    for col in categorical_cols:
        if col in combined.columns:
            encoder = LabelEncoder()
            combined[col] = encoder.fit_transform(combined[col].astype(str))
    train_enc = combined.iloc[: len(train)].copy()
    test_enc = combined.iloc[len(train) :].copy()
    return train_enc, test_enc


def make_fold_target_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    col: str,
    target: str,
    folds: GroupKFold,
    groups: pd.Series,
    smoothing: int = 20,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = train_df.copy()
    test_df = test_df.copy()
    global_mean = train_df[target].mean()
    train_encoded = np.zeros(len(train_df), dtype=float)
    test_encoded = np.zeros(len(test_df), dtype=float)

    for train_idx, val_idx in folds.split(train_df, train_df[target], groups):
        fold_train = train_df.iloc[train_idx]
        stats = fold_train.groupby(col)[target].agg(['mean', 'count'])
        stats['smoothed'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
        mapping = stats['smoothed'].to_dict()
        train_encoded[val_idx] = train_df.iloc[val_idx][col].map(mapping).fillna(global_mean).to_numpy()

        full_stats = train_df.groupby(col)[target].agg(['mean', 'count'])
        full_stats['smoothed'] = (full_stats['count'] * full_stats['mean'] + smoothing * global_mean) / (full_stats['count'] + smoothing)
        full_mapping = full_stats['smoothed'].to_dict()
        test_encoded += test_df[col].map(full_mapping).fillna(global_mean).to_numpy() / folds.n_splits

    train_df[f'{col}_target_enc'] = train_encoded
    test_df[f'{col}_target_enc'] = test_encoded
    return train_df, test_df


def clip_prediction(values: np.ndarray) -> np.ndarray:
    return np.clip(values, 0.0, None)


def main() -> None:
    cfg = Config(TRAIN_PATH, TEST_PATH, OUTPUT_PATH)
    train = pd.read_csv(cfg.train_path)
    test = pd.read_csv(cfg.test_path)

    train = add_time_features(train)
    test = add_time_features(test)

    train = train.sort_values(['day', 'hour', 'minute', 'Index']).reset_index(drop=True)
    test = test.sort_values(['day', 'hour', 'minute', 'Index']).reset_index(drop=True)

    groups = train['day'].astype(int)
    te_folds = GroupKFold(n_splits=min(N_SPLITS, groups.nunique()))

    for column, smooth in [
        ('geohash', 15),
        ('geo_prefix', 30),
        ('geo_prefix3', 50),
        ('geo_prefix5', 25),
        ('Weather', 40),
        ('RoadType', 40),
    ]:
        train, test = make_fold_target_encoding(train, test, column, 'demand', te_folds, groups, smoothing=smooth)

    train['hour_x_geo'] = train['hour'] * train['geohash_target_enc']
    test['hour_x_geo'] = test['hour'] * test['geohash_target_enc']
    train['temp_x_weather'] = train['Temperature'] * train['Weather_target_enc']
    test['temp_x_weather'] = test['Temperature'] * test['Weather_target_enc']
    train['rush_x_geo'] = train['rush_hour'] * train['geohash_target_enc']
    test['rush_x_geo'] = test['rush_hour'] * test['geohash_target_enc']
    train['time_x_geo_prefix'] = train['time_minutes'] * train['geo_prefix_target_enc']
    test['time_x_geo_prefix'] = test['time_minutes'] * test['geo_prefix_target_enc']

    categorical_cols = ['geohash', 'geo_prefix', 'geo_prefix3', 'geo_prefix5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'time_period', 'temp_bin']
    train, test = encode_categoricals(train, test, categorical_cols)

    drop_cols = ['Index', 'demand', 'timestamp']
    feature_cols = [col for col in train.columns if col not in drop_cols]

    X = train[feature_cols].copy()
    y = train['demand'].copy()
    X_test = test[feature_cols].copy()

    medians = X.median(numeric_only=True)
    X = X.fillna(medians).fillna(0)
    X_test = X_test.fillna(medians).fillna(0)

    cat_features = [feature_cols.index(col) for col in categorical_cols if col in feature_cols]
    model_folds = GroupKFold(n_splits=min(N_SPLITS, groups.nunique()))

    lgb_params = {
        'n_estimators': 3000,
        'learning_rate': 0.02,
        'max_depth': -1,
        'num_leaves': 128,
        'min_child_samples': 20,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 0.5,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1,
    }
    xgb_params = {
        'n_estimators': 3500,
        'learning_rate': 0.02,
        'max_depth': 8,
        'min_child_weight': 2,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.1,
        'reg_lambda': 0.5,
        'gamma': 0.0,
        'tree_method': 'hist',
        'objective': 'reg:squarederror',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': 0,
    }
    cat_params = {
        'iterations': 3500,
        'learning_rate': 0.02,
        'depth': 8,
        'l2_leaf_reg': 4.0,
        'loss_function': 'RMSE',
        'random_seed': RANDOM_STATE,
        'verbose': 0,
        'allow_writing_files': False,
    }

    oof_lgb = np.zeros(len(X), dtype=float)
    oof_xgb = np.zeros(len(X), dtype=float)
    oof_cat = np.zeros(len(X), dtype=float)
    pred_lgb = np.zeros(len(X_test), dtype=float)
    pred_xgb = np.zeros(len(X_test), dtype=float)
    pred_cat = np.zeros(len(X_test), dtype=float)

    for fold, (train_idx, val_idx) in enumerate(model_folds.split(X, y, groups), start=1):
        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        lgb_model = lgb.LGBMRegressor(**lgb_params)
        lgb_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)],
        )
        oof_lgb[val_idx] = lgb_model.predict(X_val)
        pred_lgb += lgb_model.predict(X_test) / model_folds.n_splits

        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        oof_xgb[val_idx] = xgb_model.predict(X_val)
        pred_xgb += xgb_model.predict(X_test) / model_folds.n_splits

        cat_model = CatBoostRegressor(**cat_params)
        cat_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_features,
            use_best_model=True,
        )
        oof_cat[val_idx] = cat_model.predict(X_val)
        pred_cat += cat_model.predict(X_test) / model_folds.n_splits

        print(
            f'Fold {fold}: '
            f'LGB={r2_score(y_val, oof_lgb[val_idx]):.5f} '
            f'XGB={r2_score(y_val, oof_xgb[val_idx]):.5f} '
            f'CAT={r2_score(y_val, oof_cat[val_idx]):.5f}'
        )

    base_oof = np.column_stack([oof_lgb, oof_xgb, oof_cat])
    base_test = np.column_stack([pred_lgb, pred_xgb, pred_cat])

    meta = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    meta.fit(base_oof, y.values)
    oof_meta = meta.predict(base_oof)
    test_pred = meta.predict(base_test)

    base_scores = {
        'lgb': r2_score(y, oof_lgb),
        'xgb': r2_score(y, oof_xgb),
        'cat': r2_score(y, oof_cat),
        'meta': r2_score(y, oof_meta),
    }
    print('OOF scores:', {k: round(v, 6) for k, v in base_scores.items()})
    print('Meta coefficients:', meta.coef_)

    final_pred = clip_prediction(test_pred)
    submission = pd.DataFrame({'Index': test['Index'].values, 'demand': final_pred})
    submission.to_csv(cfg.output_path, index=False)
    print(f'Saved {cfg.output_path} with shape {submission.shape}')


main()

: 